# Faz 1 — Veri Hazırlık ve Manifest

**Amaç:** TR_ABDOMEN_RAD_EMERGENCY veri setini incele, `Bilgi.xlsx`'teki ham annotasyonları doğrula, 16 ham etiketi 6 üst sınıfa eşle ve `manifest.csv` üret.

**Adımlar**
1. Ortam ve yol kontrolü
2. `Bilgi.xlsx` keşfi (TRAIININGDATA + COMPETITIONDATA)
3. Ham etiket → 6 üst sınıf eşleme istatistikleri
4. Tek vaka üzerinde DICOM yükleme + HU pencereleme görseli
5. `build_manifest()` → `outputs/splits/manifest.csv`
6. Manifest doğrulama

## 1. Ortam ve Yol Kontrolü

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))
YARISMA_DIR = Path(os.environ.get("ABDOMEN_TEST_DIR", DATA_ROOT / "Test Verisi"))


sys.path.insert(0, str(CODE))

print("DATA_ROOT   :", DATA_ROOT, "→ var?", DATA_ROOT.exists())
print("EGITIM_DIR  :", EGITIM_DIR, "→ var?", EGITIM_DIR.exists())
print("Bilgi.xlsx  :", (DATA_ROOT / "Bilgi.xlsx").exists())
print("CODE/src    :", (CODE / "src").exists())
egitim_cases = sorted([d for d in EGITIM_DIR.iterdir() if d.is_dir()]) if EGITIM_DIR.exists() else []
print(f"\nEğitim vaka klasörü sayısı: {len(egitim_cases)}")
print("İlk 5 vaka:", [d.name for d in egitim_cases[:5]])

## 2. Bilgi.xlsx Keşfi

In [ ]:
import pandas as pd
xl = pd.ExcelFile(DATA_ROOT / "Bilgi.xlsx")
print("Sayfalar:", xl.sheet_names)
train_df = pd.read_excel(xl, sheet_name="TRAIININGDATA")
comp_df  = pd.read_excel(xl, sheet_name="COMPETITIONDATA")
print(f"\nTRAIININGDATA   : {len(train_df):,} annotasyon, {train_df['Case Number'].nunique()} vaka")
print(f"COMPETITIONDATA : {len(comp_df):,} annotasyon, {comp_df['Case Number'].nunique()} vaka")
print(f"Sütunlar: {list(train_df.columns)}")
train_df.head(5)

In [ ]:
print('=== TRAIININGDATA — Type ===')
print(train_df['Type'].value_counts())
print('\n=== COMPETITIONDATA — Type ===')
print(comp_df['Type'].value_counts())

In [ ]:
print('=== TRAIININGDATA — Class (ilk 20) ===')
print(train_df['Class'].value_counts().head(20))

## 3. Ham Etiket → 6 Üst Sınıf Eşleme

In [ ]:
from src.config import RAW_PATHOLOGY_TO_SUPER, SUPER_CLASSES, ANATOMICAL_TO_ID
print('6 ÜST SINIF:')
for i, s in enumerate(SUPER_CLASSES):
    print(f'  {i}: {s}')
print('\nHam patoloji → üst sınıf:')
for raw, sid in RAW_PATHOLOGY_TO_SUPER.items():
    print(f'  {raw:50s} → {sid} ({SUPER_CLASSES[sid]})')

In [ ]:
all_classes = pd.concat([train_df['Class'], comp_df['Class']]).value_counts()
mapped = {c: cnt for c, cnt in all_classes.items() if c in RAW_PATHOLOGY_TO_SUPER}
anat   = {c: cnt for c, cnt in all_classes.items() if c in ANATOMICAL_TO_ID}
unmapped = {c: cnt for c, cnt in all_classes.items()
            if c not in RAW_PATHOLOGY_TO_SUPER and c not in ANATOMICAL_TO_ID}
print(f"Patoloji (eşlenen): {sum(mapped.values()):,} annot, {len(mapped)} sınıf")
print(f"Anatomik:           {sum(anat.values()):,} annot, {len(anat)} sınıf")
print(f"Eşlenmeyen:         {sum(unmapped.values()):,} annot")
for c, n in list(unmapped.items())[:10]:
    print(f'  {c}: {n}')

## 4. Tek Vaka Görsel Doğrulama (HU Pencereleme)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.dicom_utils import read_dicom, dicom_to_hu, hu_to_three_channel
from src.config import DEFAULT_WINDOWS

sample_row = train_df[train_df['Type'] == 'Bounding Box'].iloc[0]
case = int(sample_row['Case Number'])
img_id = int(sample_row['Image Id'])
dcm_path = EGITIM_DIR / str(case) / f"{img_id}.dcm"
print(f"Vaka {case}, kesit {img_id} | sınıf: {sample_row['Class']} | bbox: {sample_row['Data']}")
print(f"DICOM yolu: {dcm_path} (var: {dcm_path.exists()})")

ds = read_dicom(dcm_path)
hu = dicom_to_hu(ds)
rgb = hu_to_three_channel(hu, DEFAULT_WINDOWS)
print(f"HU shape: {hu.shape}, min={hu.min():.0f} max={hu.max():.0f}")
print(f"3 kanallı çıktı: {rgb.shape}")

In [ ]:
bbox_str = str(sample_row['Data'])
x1y1, x2y2 = bbox_str.split('-')
x1, y1 = [int(v) for v in x1y1.split(',')]
x2, y2 = [int(v) for v in x2y2.split(',')]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(hu, cmap='gray', vmin=-200, vmax=400)
axes[0].set_title(f'HU [-200,400]\nVaka {case}, {img_id}')
for i, w in enumerate(DEFAULT_WINDOWS):
    axes[i+1].imshow(rgb[..., i], cmap='gray')
    axes[i+1].set_title(f'{w.name}\nWL={w.level}, WW={w.width}')
for ax in axes:
    ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                    edgecolor='red', facecolor='none'))
    ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Manifest Üretimi

In [ ]:
from src.config import SPLIT_DIR
from src.preprocessing import build_manifest

out_path: Path = SPLIT_DIR / "manifest.csv"
manifest_path = out_path if out_path.exists() else build_manifest(out_path)
print('Manifest yazıldı:', manifest_path)

In [ ]:
manifest = pd.read_csv(manifest_path)
print(f'Toplam kesit: {len(manifest):,}')
print(f'Tekil vaka: {manifest["case"].nunique()}')
print(f'\nKaynak dağılımı:')
print(manifest['source'].value_counts())
manifest['has_path'] = manifest['super_labels'].fillna('').astype(str).str.len() > 0
manifest['has_bbox'] = manifest['bboxes'].fillna('').astype(str).str.len() > 0
print(f'\nÜst sınıf etiketli: {manifest["has_path"].sum():,}')
print(f'BBox\'lu kesit:      {manifest["has_bbox"].sum():,}')
manifest.head(5)

In [ ]:
from collections import Counter
cnt = Counter()
for v in manifest['super_labels'].fillna('').astype(str):
    if not v: continue
    for s in v.split(';'):
        if s.strip(): cnt[int(s)] += 1
print('Üst sınıf bazında BBox kesit sayısı:')
for sid, c in sorted(cnt.items()):
    print(f'  {sid} ({SUPER_CLASSES[sid]}): {c:,}')

## 6. Faz 1 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| Manifest | `outputs/splits/manifest.csv` |

**Sonraki adım:** `Faz2_Split_Onisleme_1fold.ipynb`